# Multi-Agent System

**Architecture:**
```
Orchestrator (Sonnet)  <- smart, coordinates the full run
     | tool_use
  +--+------------------+
Planner (Haiku)   Executor (Haiku)   Critic (Haiku)
     ^ cheap workers, each a fresh Haiku call
```

**Cost levers used:**
- Haiku for all worker agents (~12x cheaper than Sonnet)
- Prompt caching on system prompts (up to 90% off repeated tokens)
- Tight `max_tokens` per role
- Token usage logged after every call

**Setup:**
```bash
pip install anthropic
export ANTHROPIC_API_KEY=sk-ant-...
```

## Imports & Config

In [5]:
%pip install anthropic python-dotenv


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [6]:
from dotenv import load_dotenv
load_dotenv()

True

In [7]:
import json
import os
import anthropic

ORCHESTRATOR_MODEL  = "claude-sonnet-4-6"
WORKER_MODEL        = "claude-haiku-4-5-20251001"

MAX_ORCHESTRATOR_TOKENS = 2048
MAX_WORKER_TOKENS       = 1024
MAX_ITERATIONS          = 10       # safety cap on the agentic loop

client = anthropic.Anthropic()

## Cost Tracker

In [8]:
# Prices per million tokens (May 2025)
COST_PER_M = {
    ORCHESTRATOR_MODEL: {"input": 3.00,  "output": 15.00},
    WORKER_MODEL:       {"input": 0.80,  "output":  4.00},
}

total_cost_usd = 0.0

def log_usage(model: str, usage: anthropic.types.Usage, label: str) -> None:
    global total_cost_usd
    rates = COST_PER_M.get(model, {"input": 3.0, "output": 15.0})

    cache_read    = getattr(usage, "cache_read_input_tokens", 0) or 0
    cache_written = getattr(usage, "cache_creation_input_tokens", 0) or 0
    billed_input  = usage.input_tokens - cache_read          # cache reads cost ~10% of normal

    cost = (
        (billed_input / 1_000_000) * rates["input"]
        + (cache_read / 1_000_000) * rates["input"] * 0.10  # 90% discount on cache hits
        + (usage.output_tokens / 1_000_000) * rates["output"]
    )
    total_cost_usd += cost

    print(
        f"  📊 {label} | in={usage.input_tokens}"
        + (f" (cache_hit={cache_read}, cache_write={cache_written})" if cache_read or cache_written else "")
        + f" | out={usage.output_tokens} | ${cost:.5f}"
    )

## System Prompts

In [9]:
ORCHESTRATOR_SYSTEM = """\
You are an Orchestrator that coordinates three specialized AI agents to complete tasks.

Agents available (call via tools):
  call_planner   — breaks a goal into ordered subtasks
  call_executor  — executes one subtask and returns concrete output
  call_critic    — reviews output against criteria, returns score + feedback

Standard workflow:
  1. call_planner  → get subtask list
  2. call_executor → for each subtask (pass context from completed steps)
  3. call_critic   → on the final combined output
  4. If critic score < 7, re-execute failing subtasks with its feedback
  5. Return a clear final answer to the user

Be efficient. Do not call agents unnecessarily.\
"""

PLANNER_SYSTEM = """\
You are a Planner. Decompose the given goal into clear, ordered, actionable subtasks.

Rules:
  - Output ONLY a numbered list. No preamble, no explanation.
  - Each subtask: specific, independently actionable, 1-2 sentences.
  - 3-6 subtasks is ideal. Never more than 8.\
"""

EXECUTOR_SYSTEM = """\
You are an Executor. You receive one subtask and must complete it thoroughly.

Rules:
  - Think step by step internally, then output only the concrete result.
  - Be specific — no vague summaries.
  - If context from prior steps is provided, use it.\
"""

CRITIC_SYSTEM = """\
You are a Critic. Evaluate the provided output against the stated criteria.

Return ONLY a JSON object in this exact shape — no other text:
{
  "score": <1-10>,
  "passed": <true|false>,
  "issues": ["<issue>", ...],
  "suggestions": ["<suggestion>", ...]
}\
"""

## Worker Agent Calls

All workers use Haiku with prompt caching on the system prompt.

In [10]:
def _worker_call(label: str, system: str, user_message: str) -> str:
    """Single Haiku call with prompt caching on the system prompt."""
    response = client.messages.create(
        model=WORKER_MODEL,
        max_tokens=MAX_WORKER_TOKENS,
        system=[{
            "type": "text",
            "text": system,
            "cache_control": {"type": "ephemeral"},   # caches system prompt
        }],
        messages=[{"role": "user", "content": user_message}],
    )
    log_usage(WORKER_MODEL, response.usage, label)
    return response.content[0].text


def call_planner(goal: str) -> str:
    return _worker_call("Planner", PLANNER_SYSTEM, f"Goal: {goal}")


def call_executor(subtask: str, context: str = "") -> str:
    msg = f"Subtask: {subtask}"
    if context:
        msg += f"\n\nContext from previous steps:\n{context}"
    return _worker_call("Executor", EXECUTOR_SYSTEM, msg)


def call_critic(output: str, criteria: str) -> str:
    return _worker_call(
        "Critic",
        CRITIC_SYSTEM,
        f"Output to review:\n{output}\n\nCriteria:\n{criteria}",
    )

## Tool Definitions

Schema exposed to the Orchestrator.

In [11]:
AGENT_TOOLS = [
    {
        "name": "call_planner",
        "description": "Decompose a high-level goal into ordered, actionable subtasks.",
        "input_schema": {
            "type": "object",
            "properties": {
                "goal": {"type": "string", "description": "The goal to decompose."},
            },
            "required": ["goal"],
        },
    },
    {
        "name": "call_executor",
        "description": (
            "Execute a single subtask and return concrete output. "
            "Pass context from already-completed subtasks to help this one."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "subtask": {"type": "string", "description": "The subtask to execute."},
                "context": {
                    "type": "string",
                    "description": "Relevant output from previous completed subtasks.",
                },
            },
            "required": ["subtask"],
        },
    },
    {
        "name": "call_critic",
        "description": (
            "Review output against criteria. Returns JSON with score (1-10), "
            "passed flag, issues, and suggestions."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "output":   {"type": "string", "description": "The output to review."},
                "criteria": {"type": "string", "description": "What to evaluate against."},
            },
            "required": ["output", "criteria"],
        },
    },
]

## Tool Dispatch

In [12]:
def dispatch(tool_name: str, tool_input: dict) -> str:
    match tool_name:
        case "call_planner":
            return call_planner(tool_input["goal"])
        case "call_executor":
            return call_executor(tool_input["subtask"], tool_input.get("context", ""))
        case "call_critic":
            return call_critic(tool_input["output"], tool_input["criteria"])
        case _:
            return json.dumps({"error": f"Unknown agent: {tool_name}"})

## Orchestrator Loop

In [13]:
def run(goal: str) -> str:
    """Run the multi-agent loop for a given goal. Returns the final answer."""
    global total_cost_usd
    total_cost_usd = 0.0  # reset cost for each run

    print(f"\n{'═'*60}")
    print(f"Goal: {goal}")
    print(f"{'═'*60}")

    messages = [{"role": "user", "content": goal}]

    for iteration in range(1, MAX_ITERATIONS + 1):
        print(f"\n[Iteration {iteration}]")

        response = client.messages.create(
            model=ORCHESTRATOR_MODEL,
            max_tokens=MAX_ORCHESTRATOR_TOKENS,
            system=ORCHESTRATOR_SYSTEM,
            tools=AGENT_TOOLS,
            messages=messages,
        )
        log_usage(ORCHESTRATOR_MODEL, response.usage, "Orchestrator")

        # Done: extract and return final text
        if response.stop_reason == "end_turn":
            for block in response.content:
                if hasattr(block, "text"):
                    print(f"\n{'─'*60}")
                    print(f"Total cost: ${total_cost_usd:.5f}")
                    print(f"{'─'*60}")
                    return block.text
            return "No response generated."

        # Tool use: dispatch to worker agents
        if response.stop_reason == "tool_use":
            # 1. Append assistant turn (includes tool_use blocks)
            messages.append({"role": "assistant", "content": response.content})

            # 2. Execute every tool call in this round, collect results
            tool_results = []
            for block in response.content:
                if block.type != "tool_use":
                    continue
                print(f"  -> {block.name}({json.dumps(block.input)[:80]}...)")
                result = dispatch(block.name, block.input)
                print(f"  <- {len(result)} chars returned")
                tool_results.append({
                    "type":        "tool_result",
                    "tool_use_id": block.id,
                    "content":     result,
                })

            # 3. Append all tool results in one user turn
            messages.append({"role": "user", "content": tool_results})

        else:
            print(f"  Unexpected stop_reason: {response.stop_reason}")
            break

    return f"Reached max iterations ({MAX_ITERATIONS}). Check logs above for partial output."

## Run

Set your goal below and execute the cell.

In [14]:
goal = "Write a concise market analysis for electric vehicles in Canada"

result = run(goal)

print(f"\n{'═'*60}")
print("FINAL RESULT")
print(f"{'═'*60}")
print(result)


════════════════════════════════════════════════════════════
Goal: Write a concise market analysis for electric vehicles in Canada
════════════════════════════════════════════════════════════

[Iteration 1]
  📊 Orchestrator | in=961 | out=65 | $0.00386
  -> call_planner({"goal": "Write a concise market analysis for electric vehicles in Canada"}...)
  📊 Planner | in=103 | out=151 | $0.00069
  <- 674 chars returned

[Iteration 2]
  📊 Orchestrator | in=1184 | out=454 | $0.01036
  -> call_executor({"subtask": "Research current EV market size, growth rate, and sales figures in ...)
  📊 Executor | in=117 | out=392 | $0.00166
  <- 1250 chars returned
  -> call_executor({"subtask": "Identify and analyze the top EV manufacturers and models selling in...)
  📊 Executor | in=112 | out=560 | $0.00233
  <- 1808 chars returned
  -> call_executor({"subtask": "Examine key drivers of EV adoption in Canada: government incentives...)
  📊 Executor | in=117 | out=1010 | $0.00413
  <- 4301 chars returned
  